# Qiskit Port Tutorial: Major Qulacs–Qiskit Comparisons

This tutorial demonstrates the major validated components of the Qiskit port of the Anderson Impurity Model (AIM) workflow.

The notebook uses the original Qulacs implementation as the reference and compares it directly with the Qiskit implementation at each major stage.

## Comparisons included

1. Shared AIM system initialization
2. Shifted Hamiltonian and \(H^2\)
3. Generic symmetry-preserving ansatz
4. Variational-state reconstruction
5. Particle-addition and particle-removal Krylov states
6. Initial Lanczos coefficients \(a_0\) and \(b_1\)
7. Lanczos cost function
8. One complete variational Lanczos iteration

The full multi-iteration Green's-function workflow and `dmft.py` backend integration remain under development.

## 0. Setup

Clone the fork and check out the Qiskit development branch:

```bash
git clone https://github.com/aporter7/qcaob-aim.git
cd qcaob-aim
git checkout qiskit-vqe-port-local
```

Create and activate a virtual environment:

```bash
python3 -m venv qiskit-clean
source qiskit-clean/bin/activate
```

Install the repository dependencies:

```bash
python -m pip install -r requirements.txt
python -m pip install qiskit qiskit-aer jupyterlab
```

Launch Jupyter from the repository root:

```bash
python -m jupyter lab
```

Open this notebook and run the cells in order.

## 1. Imports

The original repository modules provide the Qulacs reference implementation.  
The `qiskit_port` package contains the converted Qiskit functions.

In [1]:
import importlib
import numpy as np

from scipy.optimize import minimize
from qulacs import QuantumState
from qiskit.quantum_info import Statevector

import anderson_impurity_model as aim
import dmft
import vqe

import qiskit_port.qiskit_ansatz as qa
import qiskit_port.qiskit_vqe as qv

importlib.reload(qa)
importlib.reload(qv)

print("Imports complete.")

Imports complete.


## 2. Initialize the Shared AIM System

This cell initializes one real AIM system using the original repository workflow.

The returned model objects are shared inputs for both implementations:

- qubit Hamiltonian
- spin-up and spin-down registers
- connected interaction graphs
- impurity orbital
- number of orbitals

This step does not run VQE.

In [2]:
system_size = 3
seed = 0
vqe_depth = 1

(
    impurity_orbital,
    test_model,
    n_orbitals,
    up_qubit_indices,
    down_qubit_indices,
    connected_graphs,
    qubit_hamiltonian,
) = dmft.initialize_system(system_size, seed)

print("System size:", system_size)
print("Seed:", seed)
print("Number of orbitals:", n_orbitals)
print("Spin-up qubits:", up_qubit_indices)
print("Spin-down qubits:", down_qubit_indices)
print("Connected graph keys:", list(connected_graphs.keys()))
print("Number of Hamiltonian terms:", len(qubit_hamiltonian.terms))

System size: 3
Seed: 0
Number of orbitals: 3
Spin-up qubits: [0, 1, 2]
Spin-down qubits: [3, 4, 5]
Connected graph keys: ['graph_full', 'graph_up', 'graph_down', 'graph_stitch']
Number of Hamiltonian terms: 16


## 3. Major Comparison 1: Shifted Hamiltonian and \(H^2\)

The original function creates Qulacs operator objects.  
The Qiskit-port function creates equivalent OpenFermion/NumPy matrices.

The shifted Hamiltonian is

\[
H = H_{\mathrm{AIM}}-E_{\mathrm{shift}}I.
\]

The squared Hamiltonian \(H^2\) is required for the off-diagonal Lanczos coefficients.

For this component-level comparison, `-1.0` is used as a common test shift. It is not being presented as the physical ground-state energy.

The expectation values of \(H\) and \(H^2\) are compared over every computational basis state.

In [3]:
vqe_energy_shift = -1.0

qulacs_H, qulacs_H2, n_qubits_qulacs = vqe.create_qulacs_hamiltonian(
    qubit_hamiltonian,
    vqe_energy_shift,
)

Hmat, H2mat, n_qubits_qiskit = qv.create_qiskit_hamiltonian_matrix(
    qubit_hamiltonian,
    vqe_energy_shift,
)

assert n_qubits_qulacs == n_qubits_qiskit
n_qubits = n_qubits_qulacs

max_diff_H = 0.0
max_diff_H2 = 0.0

for basis in range(2**n_qubits):
    qs = QuantumState(n_qubits)
    qs.set_computational_basis(basis)

    raw_state = np.zeros(2**n_qubits, dtype=complex)
    raw_state[basis] = 1.0

    # OpenFermion matrices require the reversed-index representation.
    ordered_state = qv.qulacs_to_python_ordering_qiskit(
        raw_state,
        n_qubits,
    )

    qulacs_exp_H = qulacs_H.get_expectation_value(qs).real
    qiskit_exp_H = np.vdot(
        ordered_state,
        Hmat @ ordered_state,
    ).real

    qulacs_exp_H2 = qulacs_H2.get_expectation_value(qs).real
    qiskit_exp_H2 = np.vdot(
        ordered_state,
        H2mat @ ordered_state,
    ).real

    max_diff_H = max(
        max_diff_H,
        abs(qulacs_exp_H - qiskit_exp_H),
    )
    max_diff_H2 = max(
        max_diff_H2,
        abs(qulacs_exp_H2 - qiskit_exp_H2),
    )

print("Maximum H expectation-value difference:", max_diff_H)
print("Maximum H^2 expectation-value difference:", max_diff_H2)

assert max_diff_H < 1e-10
assert max_diff_H2 < 1e-10

print("PASS: Qiskit Hamiltonian matrices match Qulacs.")

Maximum H expectation-value difference: 5.329070518200751e-15
Maximum H^2 expectation-value difference: 8.526512829121202e-14
PASS: Qiskit Hamiltonian matrices match Qulacs.


## 4. Shared Variational Parameters and Occupation Sector

The same occupation sector, ansatz depth, connected graphs, and parameter vector are used by both backends.

These parameters are shared validation parameters. They are not claimed to be optimized ground-state parameters.

In [4]:
n_up = 1
n_down = 1

initial_occupations_indices = vqe.get_initial_occupations_indices(
    up_qubit_indices,
    down_qubit_indices,
    n_up,
    n_down,
)

n_edges = connected_graphs["graph_full"].number_of_edges()
n_params = vqe_depth * (n_edges + n_qubits)

np.random.seed(0)
theta = np.random.uniform(
    0.0,
    2.0 * np.pi,
    size=n_params,
)

print("Number of qubits:", n_qubits)
print("Number of parameters:", n_params)
print("Initial occupations:", initial_occupations_indices)

Number of qubits: 6
Number of parameters: 13
Initial occupations: [0, 3]


## 5. Major Comparison 2: Generic Symmetry-Preserving Ansatz

The original Qulacs ansatz and the Qiskit ansatz receive identical parameters and graph inputs.

The resulting statevectors are compared directly. No bit-order reversal is required for a direct Qulacs-circuit-state versus Qiskit-circuit-state comparison.

A global phase correction is applied because global phase has no physical effect.

In [11]:
# Original Qulacs circuit and state
symm_qulacs = aim.SymmQulacsVqeEmulator(qulacs_H)

qulacs_circuit = symm_qulacs.all_symmetry_ansatzae(
    theta=theta,
    n_layers=vqe_depth,
    initial_occupations_indices=initial_occupations_indices,
    connected_graphs=connected_graphs,
    compilation="generic",
)

qs = QuantumState(n_qubits)
qs.set_zero_state()
qulacs_circuit.update_quantum_state(qs)
qulacs_state = qs.get_vector()

# Qiskit circuit and state
qiskit_circuit = qa.all_symmetry_ansatzae_qiskit(
    theta=theta,
    n_qubits=n_qubits,
    n_layers=vqe_depth,
    initial_occupations_indices=initial_occupations_indices,
    connected_graphs=connected_graphs,
    compilation="generic",
)

qiskit_state = Statevector.from_instruction(
    qiskit_circuit
).data

overlap_value = np.vdot(qulacs_state, qiskit_state)
phase = (
    np.conj(overlap_value) / abs(overlap_value)
    if abs(overlap_value) > 0
    else 1.0
)
qiskit_state_corrected = phase * qiskit_state

ansatz_difference = np.max(
    np.abs(qulacs_state - qiskit_state_corrected)
)

print("State overlap magnitude:", abs(overlap_value))
print("Phase-corrected max difference:", ansatz_difference)

assert ansatz_difference < 1e-10


State overlap magnitude: 0.9999999999999984
Phase-corrected max difference: 2.482534153247273e-16


## 6. Major Comparison 3: Variational-State Reconstruction

`construct_vqe_gs()` rebuilds a variational state from a stored parameter vector.

This comparison validates reconstruction logic only. The shared parameters are not being presented as optimized ground-state parameters.

In [13]:
qulacs_gs, _ = vqe.construct_vqe_gs(
    n_qubits=n_qubits,
    qulacs_ham=qulacs_H,
    up_qubit_indices=up_qubit_indices,
    down_qubit_indices=down_qubit_indices,
    vqe_depth=vqe_depth,
    connected_graphs=connected_graphs,
    minimum_angles=theta,
    vqe_nu=n_up,
    vqe_nd=n_down,
)

qiskit_gs, qiskit_gs_array = qv.construct_vqe_gs_qiskit(
    n_qubits=n_qubits,
    up_qubit_indices=up_qubit_indices,
    down_qubit_indices=down_qubit_indices,
    vqe_depth=vqe_depth,
    connected_graphs=connected_graphs,
    minimum_angles=theta,
    vqe_nu=n_up,
    vqe_nd=n_down,
)

qulacs_gs_array = qulacs_gs.get_vector()

reconstruction_overlap = np.vdot(
    qulacs_gs_array,
    qiskit_gs_array,
)
phase = (
    np.conj(reconstruction_overlap) / abs(reconstruction_overlap)
    if abs(reconstruction_overlap) > 0
    else 1.0
)
qiskit_gs_corrected = phase * qiskit_gs_array

reconstruction_difference = np.max(
    np.abs(qulacs_gs_array - qiskit_gs_corrected)
)

print("Overlap magnitude:", abs(reconstruction_overlap))
print("Phase-corrected max difference:", reconstruction_difference)
print("Qulacs norm:", np.linalg.norm(qulacs_gs_array))
print("Qiskit norm:", np.linalg.norm(qiskit_gs_array))

assert reconstruction_difference < 1e-10



Overlap magnitude: 0.9999999999999984
Phase-corrected max difference: 2.482534153247273e-16
Qulacs norm: 0.9999999999999991
Qiskit norm: 0.999999999999999


## Generic Ansatz Validation Result

The Qiskit implementation of the generic symmetry-preserving ansatz was compared directly against the original Qulacs implementation using a real \(N=3\) AIM system.

Both circuits used:

- the same connected AIM graphs,
- the same initial occupation indices,
- the same number of layers, and
- the same parameter vector.

The resulting statevectors were compared directly. A global phase correction was applied because global phase does not change the physical quantum state.

Results:

- Overlap magnitude: `0.9999999999999984`
- Phase-corrected maximum difference: `2.482534153247273e-16`

These values indicate agreement at machine precision.

Therefore, the Qiskit generic ansatz reproduces the original Qulacs generic ansatz for this test case.

In [17]:
import importlib
import qiskit_port.qiskit_ansatz as qa

importlib.reload(qa)

qiskit_circuit = qa.all_symmetry_ansatzae_qiskit(
    theta=theta,
    n_qubits=n_qubits,
    n_layers=vqe_depth,
    initial_occupations_indices=initial_occupations_indices,
    connected_graphs=connected_graphs,
    compilation="generic",
)

print(qiskit_circuit)

     ┌───┐┌───┐┌───┐     ┌─────────────┐      ┌───┐ ┌─────┐     ┌───┐»
q_0: ┤ X ├┤ S ├┤ H ├──■──┤ Ry(-3.4483) ├──■───┤ H ├─┤ Sdg ├─────┤ X ├»
     ├───┤└───┘└───┘┌─┴─┐├─────────────┤┌─┴─┐┌┴───┴┐└┬───┬┘┌───┐└─┬─┘»
q_1: ┤ S ├──────────┤ X ├┤ Ry(-3.4483) ├┤ X ├┤ Sdg ├─┤ S ├─┤ H ├──┼──»
     ├───┤          └───┘└─────────────┘└───┘└─────┘ └───┘ └───┘  │  »
q_2: ┤ S ├────────────────────────────────────────────────────────┼──»
     ├───┤┌───┐┌───┐     ┌─────────────┐      ┌───┐ ┌─────┐       │  »
q_3: ┤ X ├┤ S ├┤ H ├──■──┤ Ry(-3.7873) ├──■───┤ H ├─┤ Sdg ├───────■──»
     ├───┤└───┘└───┘┌─┴─┐├─────────────┤┌─┴─┐┌┴───┴┐└┬───┬┘┌───┐     »
q_4: ┤ S ├──────────┤ X ├┤ Ry(-3.7873) ├┤ X ├┤ Sdg ├─┤ S ├─┤ H ├──■──»
     ├───┤          └───┘└─────────────┘└───┘└─────┘ └───┘ └───┘┌─┴─┐»
q_5: ┤ S ├──────────────────────────────────────────────────────┤ X ├»
     └───┘                                                      └───┘»
«     ┌─────────────┐                ┌───┐ ┌─────────────┐                   

In [18]:
import numpy as np
import importlib

from qulacs import QuantumState
import anderson_impurity_model as aim
import dmft
import vqe

import qiskit_port.qiskit_ansatz as qa
from qiskit.quantum_info import Statevector

importlib.reload(qa)

system_size = 3
seed = 0
vqe_depth = 1

(
    impurity_orbital,
    test_model,
    n_orbitals,
    up_qubit_indices,
    down_qubit_indices,
    connected_graphs,
    qubit_hamiltonian,
) = dmft.initialize_system(system_size, seed)

# Use repo-generated qubit count
qulacs_ham, _, n_qubits = vqe.create_qulacs_hamiltonian(
    qubit_hamiltonian,
    vqe_gs_energy=-1.0,
)

# Use a real occupation sector
n_up = 1
n_down = 1

initial_occupations_indices = vqe.get_initial_occupations_indices(
    up_qubit_indices,
    down_qubit_indices,
    n_up,
    n_down,
)

n_edges = connected_graphs["graph_full"].number_of_edges()
n_params = vqe_depth * (n_edges + n_qubits)

np.random.seed(0)
theta = np.random.uniform(0, 2 * np.pi, size=n_params)

# Original Qulacs ansatz
symm_qulacs = aim.SymmQulacsVqeEmulator(qulacs_ham)
qulacs_circuit = symm_qulacs.all_symmetry_ansatzae(
    theta=theta,
    n_layers=vqe_depth,
    initial_occupations_indices=initial_occupations_indices,
    connected_graphs=connected_graphs,
    compilation="generic",
)

qs = QuantumState(n_qubits)
qs.set_zero_state()
qulacs_circuit.update_quantum_state(qs)

# Qulacs statevector directly from Qulacs
qulacs_state = qs.get_vector()

# Qiskit statevector
qiskit_state = Statevector.from_instruction(qiskit_circuit).data

# Comparison 1: raw Qulacs vs Qiskit
raw_overlap = np.vdot(qulacs_state, qiskit_state)

if abs(raw_overlap) > 0:
    # Remove any physically irrelevant global phase
    phase_correction = np.conj(raw_overlap) / abs(raw_overlap)
    qiskit_state_phase_corrected = phase_correction * qiskit_state
else:
    qiskit_state_phase_corrected = qiskit_state

raw_diff = np.max(
    np.abs(qulacs_state - qiskit_state_phase_corrected)
)



# Qulacs statevector directly from Qulacs
qulacs_state = qs.get_vector()

# Qiskit statevector
qiskit_state = Statevector.from_instruction(qiskit_circuit).data

# Compare directly
overlap_value = np.vdot(qulacs_state, qiskit_state)

if abs(overlap_value) > 0:
    phase_correction = np.conj(overlap_value) / abs(overlap_value)
    qiskit_state_phase_corrected = phase_correction * qiskit_state
else:
    qiskit_state_phase_corrected = qiskit_state

max_difference = np.max(
    np.abs(qulacs_state - qiskit_state_phase_corrected)
)

print("n_qubits:", n_qubits)
print("n_params:", n_params)
print("initial occupations:", initial_occupations_indices)
print("overlap magnitude:", abs(overlap_value))
print("phase-corrected max difference:", max_difference)

assert np.allclose(
    qulacs_state,
    qiskit_state_phase_corrected,
    atol=1e-10,
)

print("Qiskit generic ansatz matches Qulacs generic ansatz.")

n_qubits: 6
n_params: 13
initial occupations: [0, 3]
overlap magnitude: 0.9999999999999984
phase-corrected max difference: 2.482534153247273e-16
Qiskit generic ansatz matches Qulacs generic ansatz.


## 7. Major Comparison 4: Particle-Removal and Particle-Addition Krylov States

The Green's-function calculation begins with two Krylov states:

\[
\phi^- \quad \text{(particle removal)}
\]

and

\[
\phi^+ \quad \text{(particle addition)}.
\]

Both implementations apply the same Jordan–Wigner parity string, occupation projector, impurity-qubit flip, and normalization.

In [14]:
(
    qulacs_phi_minus,
    _,
    qulacs_phi_minus_norm,
    qulacs_phi_plus,
    _,
    qulacs_phi_plus_norm,
    qulacs_krylov_record,
) = vqe.vqe_krylov_zero_state(
    vqe_gs=qulacs_gs,
    impurity_orbital=impurity_orbital,
    n_qubits=n_qubits,
)

(
    qiskit_phi_minus,
    qiskit_phi_minus_array,
    qiskit_phi_minus_norm,
    qiskit_phi_plus,
    qiskit_phi_plus_array,
    qiskit_phi_plus_norm,
    qiskit_krylov_record,
) = qv.vqe_krylov_zero_state_qiskit(
    vqe_gs=qiskit_gs,
    impurity_orbital=impurity_orbital,
    n_qubits=n_qubits,
)

def phase_corrected_comparison(reference, candidate):
    ov = np.vdot(reference, candidate)
    phase = np.conj(ov) / abs(ov) if abs(ov) > 0 else 1.0
    corrected = phase * candidate
    difference = np.max(np.abs(reference - corrected))
    return abs(ov), difference

minus_overlap, minus_difference = phase_corrected_comparison(
    qulacs_phi_minus.get_vector(),
    qiskit_phi_minus_array,
)

plus_overlap, plus_difference = phase_corrected_comparison(
    qulacs_phi_plus.get_vector(),
    qiskit_phi_plus_array,
)

print("Phi minus norm difference:", abs(
    qulacs_phi_minus_norm - qiskit_phi_minus_norm
))
print("Phi minus overlap:", minus_overlap)
print("Phi minus state difference:", minus_difference)

print("\nPhi plus norm difference:", abs(
    qulacs_phi_plus_norm - qiskit_phi_plus_norm
))
print("Phi plus overlap:", plus_overlap)
print("Phi plus state difference:", plus_difference)

assert minus_difference < 1e-10
assert plus_difference < 1e-10
assert np.isclose(
    qulacs_phi_minus_norm,
    qiskit_phi_minus_norm,
    atol=1e-10,
)
assert np.isclose(
    qulacs_phi_plus_norm,
    qiskit_phi_plus_norm,
    atol=1e-10,
)



Phi minus norm difference: 1.3877787807814457e-17
Phi minus overlap: 0.9999999999999998
Phi minus state difference: 3.1333552408596407e-16

Phi plus norm difference: 2.220446049250313e-16
Phi plus overlap: 0.9999999999999998
Phi plus state difference: 1.2412670766236366e-16


## 8. Major Comparison 5: Initial Lanczos Coefficients

For the particle-removal branch, the first coefficients are

\[
a_0=\langle u_0|H|u_0\rangle
\]

and

\[
b_1=
\sqrt{
\langle u_0|H^2|u_0\rangle-a_0^2
}.
\]

The Qiskit state is reordered only for multiplication by the OpenFermion matrices.

In [15]:
a_qulacs = np.zeros(2, dtype=complex)
b_qulacs = np.zeros(2, dtype=complex)

a_qiskit = np.zeros(2, dtype=complex)
b_qiskit = np.zeros(2, dtype=complex)

a_qulacs[0] = qulacs_H.get_expectation_value(
    qulacs_phi_minus
)

b_qulacs[1] = np.sqrt(
    qulacs_H2.get_expectation_value(qulacs_phi_minus)
    - a_qulacs[0] ** 2
)

qiskit_phi_minus_ordered = qv.qulacs_to_python_ordering_qiskit(
    qiskit_phi_minus_array,
    n_qubits,
)

a_qiskit[0] = np.vdot(
    qiskit_phi_minus_ordered,
    Hmat @ qiskit_phi_minus_ordered,
)

b_qiskit[1] = np.sqrt(
    np.vdot(
        qiskit_phi_minus_ordered,
        H2mat @ qiskit_phi_minus_ordered,
    )
    - a_qiskit[0] ** 2
)

print("Qulacs a0:", a_qulacs[0])
print("Qiskit a0:", a_qiskit[0])
print("a0 difference:", abs(a_qulacs[0] - a_qiskit[0]))

print("\nQulacs b1:", b_qulacs[1])
print("Qiskit b1:", b_qiskit[1])
print("b1 difference:", abs(b_qulacs[1] - b_qiskit[1]))

assert np.isclose(a_qulacs[0], a_qiskit[0], atol=1e-10)
assert np.isclose(b_qulacs[1], b_qiskit[1], atol=1e-10)



Qulacs a0: (2.6046695965835696+0j)
Qiskit a0: (2.6046695965835687-5.551115123125783e-17j)
a0 difference: 8.899114524108741e-16

Qulacs b1: (1.9476081829905318+0j)
Qiskit b1: (1.9476081829905294+2.4525216080039415e-16j)
b1 difference: 2.4547727426609503e-15


## 9. Major Comparison 6: Lanczos Cost Function

The cost function tests whether a trial ansatz state satisfies the next-step Lanczos conditions.

It penalizes:

- an incorrect transition amplitude,
- overlap with the preceding Krylov vector,
- overlap with older Krylov vectors.

The same trial parameters and \(b_1\) are used for both backends.

In [16]:
ut_qulacs = [qulacs_phi_minus]
ut_qiskit = [qiskit_phi_minus_array]

n_up_minus = 0
n_down_minus = 1

initial_occupations_minus = vqe.get_initial_occupations_indices(
    up_qubit_indices,
    down_qubit_indices,
    n_up_minus,
    n_down_minus,
)

np.random.seed(1)
trial_params = np.random.uniform(
    0.0,
    2.0 * np.pi,
    size=n_params,
)

i = 1
compilation = "generic"

qulacs_cost = vqe.lanczos_cost_function(
    trial_params,
    qulacs_H,
    ut_qulacs,
    b_qulacs,
    i,
    vqe_depth,
    initial_occupations_minus,
    connected_graphs,
    compilation,
)

qiskit_cost = qv.lanczos_cost_function_qiskit(
    params=trial_params,
    Hmat=Hmat,
    ut=ut_qiskit,
    b=b_qiskit,
    i=i,
    n_layers=vqe_depth,
    initial_occupations_indices=initial_occupations_minus,
    connected_graphs=connected_graphs,
    n_qubits=n_qubits,
    compilation=compilation,
)

cost_difference = abs(qulacs_cost - qiskit_cost)

print("Qulacs cost:", qulacs_cost)
print("Qiskit cost:", qiskit_cost)
print("Absolute difference:", cost_difference)

assert cost_difference < 1e-10



Qulacs cost: 1.2216421007247122
Qiskit cost: 1.2216421007247087
Absolute difference: 3.552713678800501e-15


## 10. Major Comparison 7: One Variational Lanczos Iteration

This section runs one full variational Lanczos iteration for the particle-removal branch.

Each implementation:

1. starts from the same Krylov zero state,
2. calculates \(a_0\) and \(b_1\),
3. optimizes a trial state with COBYLA,
4. constructs \(u_1\),
5. calculates \(a_1\).

The deterministic coefficients \(a_0\) and \(b_1\) should agree at machine precision.

The optimized \(a_1\) values and final costs may differ slightly because the two backend evaluations can cause COBYLA to follow different numerical paths.

Depending on the installed SciPy version, the original Qulacs routine may warn that `gtol` and `eps` are unsupported COBYLA options.

In [10]:
charge_minus = 1
spin_minus = -1

np.random.seed(1)
a_qiskit_iter, b_qiskit_iter, qiskit_results, qiskit_ut = (
    qv.vqe_ideal_lanczos_iterations_qiskit(
        niter=1,
        u=qiskit_phi_minus_array,
        Hmat=Hmat,
        H2mat=H2mat,
        charge_sec=charge_minus,
        spin_sec=spin_minus,
        n_layers=vqe_depth,
        up_qubit_indices=up_qubit_indices,
        down_qubit_indices=down_qubit_indices,
        connected_graphs=connected_graphs,
        conv_tol=1e-6,
        optimizer="COBYLA",
        maxiter=10000,
        gf_gtol=5e-5,
    )
)

np.random.seed(1)
a_qulacs_iter, b_qulacs_iter, qulacs_results = (
    vqe.vqe_ideal_lanczos_iterations(
        niter=1,
        u=qulacs_phi_minus,
        h=qulacs_H,
        h2=qulacs_H2,
        charge_sec=charge_minus,
        spin_sec=spin_minus,
        n_layers=vqe_depth,
        up_qubit_indices=up_qubit_indices,
        down_qubit_indices=down_qubit_indices,
        connected_graphs=connected_graphs,
        conv_tol=1e-6,
        optimizer="COBYLA",
        maxiter=10000,
        gf_gtol=5e-5,
    )
)

print("Qulacs a coefficients:", a_qulacs_iter)
print("Qiskit a coefficients:", a_qiskit_iter)

print("\nQulacs b coefficients:", b_qulacs_iter)
print("Qiskit b coefficients:", b_qiskit_iter)

print("\na0 difference:", abs(
    a_qulacs_iter[0] - a_qiskit_iter[0]
))
print("a1 difference:", abs(
    a_qulacs_iter[1] - a_qiskit_iter[1]
))
print("b1 difference:", abs(
    b_qulacs_iter[1] - b_qiskit_iter[1]
))

print("\nQulacs final cost:", qulacs_results[1]["fun"])
print("Qiskit final cost:", qiskit_results[1]["fun"])

assert np.isclose(
    a_qulacs_iter[0],
    a_qiskit_iter[0],
    atol=1e-10,
)
assert np.isclose(
    b_qulacs_iter[1],
    b_qiskit_iter[1],
    atol=1e-10,
)

print("\nPASS: One Qiskit Lanczos iteration completed.")
print(
    "Note: optimized a1 and final cost are interpreted using "
    "optimization tolerances rather than machine-precision equality."
)

/Users/aporter2/Downloads/nlr_qc/qcaob-aim/qiskit_port/qiskit_vqe.py:512: OptimizeWarning: Unknown solver options: gtol, eps
  result = minimize(
/Users/aporter2/Downloads/nlr_qc/qcaob-aim/vqe.py:374: OptimizeWarning: Unknown solver options: gtol, eps
  lanczos_result = minimize(
/Users/aporter2/Downloads/nlr_qc/qcaob-aim/vqe.py:389: OptimizeWarning: Unknown solver options: gtol, eps
  lanczos_result = minimize(lanczos_cost_function, theta_0, args=opt_args, method=optimizer, tol=conv_tol,


Qulacs a coefficients: [2.6046696 +0.j 3.15029288+0.j]
Qiskit a coefficients: [2.6046696 -5.55111512e-17j 3.13061805+1.76941795e-16j]

Qulacs b coefficients: [0.        +0.j 1.94760818+0.j]
Qiskit b coefficients: [0.        +0.j 1.94760818+0.j]

a0 difference: 8.899114524108741e-16
a1 difference: 0.019674832534458453
b1 difference: 2.4424906541753444e-15

Qulacs final cost: 1.404265650730136e-06
Qiskit final cost: 1.5324461584387333e-06

PASS: One Qiskit Lanczos iteration completed.
Note: optimized a1 and final cost are interpreted using optimization tolerances rather than machine-precision equality.


## 11. Summary of Differences

| Component | Representation in Qulacs | Representation in Qiskit port | Validation result |
|---|---|---|---|
| Hamiltonian | `GeneralQuantumOperator` | OpenFermion/NumPy matrix | Matches after ordering correction |
| Circuit state | `QuantumState` | `Statevector` | Matches directly, up to global phase |
| Matrix-state ordering | Qulacs circuit indexing | OpenFermion matrix indexing | Bit reversal required |
| Generic ansatz | `ParametricQuantumCircuit` | `QuantumCircuit` | Matches at machine precision |
| Reconstructed state | `QuantumState` | `Statevector` | Matches at machine precision |
| Krylov zero states | Qulacs gates/projectors | NumPy/Qiskit state operations | Matches at machine precision |
| Initial coefficients | Qulacs operator expectations | NumPy matrix expectations | Matches at machine precision |
| Cost function | Qulacs state/operator calls | Qiskit state + NumPy matrices | Matches at machine precision |
| One optimized iteration | Qulacs + COBYLA | Qiskit + COBYLA | Same initial coefficients; small optimization-dependent differences |

## Key implementation rule

- Compare raw Qulacs and Qiskit circuit statevectors directly.
- Apply bit-order reversal only when a Qiskit statevector is multiplied by the OpenFermion Hamiltonian matrices.

## Remaining work

- Compare optimized Krylov-state overlap directly.
- Run multiple Lanczos iterations.
- Complete both particle-addition and particle-removal branches.
- Build the Qiskit Green's function.
- Overlay Qulacs and Qiskit Green's-function plots.
- Integrate backend selection into `dmft.py`.